## Permutation struggle
This notebook is kept as a reminder that you can spend a day investigating something in completely wrong place. And also you should never rely on anything not written in the docs. (Recall: "endianess='little'" issue on unpacking; resolved with the canonical unpacking function by Tom.)


In [100]:
import matplotlib.pyplot as plt
from itertools import product
import igraph as ig
import numpy as np
from collections import defaultdict
from bitarray.util import int2ba
import pandas as pd
import seaborn as sns
import lattice_symmetries as ls
from more_itertools import sliced
import itertools
from tqdm.auto import tqdm

import yaml

# Heisenberg Hamiltonian
# fmt: off
σ_x = np.array([ [0, 1]
               , [1, 0] ])
σ_y = np.array([ [0 , -1j]
               , [1j,   0] ])
σ_z = np.array([ [1,  0]
               , [0, -1] ])
# fmt: on
σ_p = σ_x + 1j * σ_y
σ_m = σ_x - 1j * σ_y

matrix = 0.5 * (np.kron(σ_p, σ_m) + np.kron(σ_m, σ_p)) + np.kron(σ_z, σ_z)
print(matrix)
number_spins = 6
hamming_weight = number_spins // 2
basis = ls.SpinBasis(
            ls.Group([]),
            number_spins=number_spins,
            hamming_weight=hamming_weight,
            spin_inversion=None, #FIXME: should be 1
        )

basis.build()

# mine_edges = mine_edges = [(0, 1), (0, 2), (2, 3), (1, 3), (2, 4), (4, 5), (3, 5), (4, 6), (6, 7), (5, 7), (6, 0), (7, 1), (1, 8), (3, 9), (8, 9), (5, 10), (9, 10), (7, 11), (10, 11), (11, 8), (8, 12), (9, 13), (12, 13), (10, 14), (13, 14), (11, 15), (14, 15), (15, 12), (12, 0), (13, 2), (14, 4), (15, 6)]
# their_edges = [[0, 1], [0, 4], [1, 2], [1, 5], [2, 3], [2, 6], [3, 0], [3, 7],
#               [4, 5], [4, 8], [5, 6], [5, 9], [6, 7], [6, 10], [7, 4], [7, 11],
#               [8, 9], [8, 12], [9, 10], [9, 13], [10, 11], [10, 14], [11, 8], [11, 15],
#               [12, 13], [12, 0], [13, 14], [13, 1], [14, 15], [14, 2], [15, 12], [15, 3]]

their_edges = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0), (0, 1), (0, 2), (1, 2), (0, 3), (0, 4)]

their_sites_to_mine_sites = [2, 3, 5, 1, 4, 0]

mine_edges = [(their_sites_to_mine_sites[a], their_sites_to_mine_sites[b]) for a, b in their_edges]

mine_hamiltonian = ls.Operator(basis, [ls.Interaction(matrix, mine_edges)])
their_hamiltonian = ls.Operator(basis, [ls.Interaction(matrix,their_edges)])

[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j -1.+0.j  2.+0.j  0.+0.j]
 [ 0.+0.j  2.+0.j -1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.+0.j]]


In [101]:
from random import sample

In [102]:
sample(range(6), 6)

[3, 1, 2, 5, 4, 0]

In [103]:
def unpack_state(state):
    return bin(state).removeprefix('0b').zfill(number_spins)

In [104]:
def mine_config_to_their(cfg: int, their_sites_to_mine_sites) -> int:
    unp_config = list(int2ba(int(cfg), ))
    unp_config = np.array([0] * (basis.number_spins - len(unp_config)) + unp_config)
    their_config = unp_config[their_sites_to_mine_sites]
    return np.uint64(sum(2 ** i * b for i, b in enumerate(their_config[::-1])))

In [105]:
def get_basises_df(their_sites_to_mine_sites):
    basises_df = (pd.DataFrame(dict(mine_basis=basis.states,
                                    mine_unpacked=[unpack_state(x) for x in basis.states],
                                    their_basis=[mine_config_to_their(x, their_sites_to_mine_sites) for x in basis.states]))
                  .assign(their_unpacked=lambda x: [unpack_state(y) for y in x['their_basis']])
                  .merge(pd.DataFrame(dict(their_basis=basis.states)).reset_index(), on="their_basis", how='outer')
                 )
    return basises_df

In [106]:
their_ham_dense = their_hamiltonian.to_csr().todense()
mine_ham_dense = mine_hamiltonian.to_csr().todense()

In [107]:
def permute_hamiltonian(ham, permutation):
    return ham[permutation, :][:, permutation]

In [108]:
basises_df = get_basises_df(list(their_sites_to_mine_sites))
(permute_hamiltonian(their_ham_dense, basises_df['index']) == mine_ham_dense).all()

False

In [109]:
def _():
    for their_sites_to_mine_sites in tqdm(list(itertools.permutations(range(number_spins)))):
        #print(their_sites_to_mine_sites)
        basises_df = get_basises_df(list(their_sites_to_mine_sites))
        if (permute_hamiltonian(their_ham_dense, basises_df['index']) == mine_ham_dense).all():
            print(their_sites_to_mine_sites)
_()

 94%|█████████▍| 675/720 [00:02<00:00, 240.67it/s]

(5, 1, 4, 0, 2, 3)


100%|██████████| 720/720 [00:02<00:00, 240.96it/s]


In [110]:
(0, 1, 2, 4, 5, 3)
(0, 1, 5, 4, 2, 3)

(0, 1, 5, 4, 2, 3)

In [49]:
their_sites_to_mine_sites

[2, 0, 1, 3, 4, 5]

In [50]:
def inverse_subs(x):
    return [x.index(y) for y in range(len(x))]

In [51]:
inverse_subs(their_sites_to_mine_sites)

[1, 2, 0, 3, 4, 5]

In [20]:
get_basises_df([1, 2, 3, 4, 5, 0])

,mine_basis,mine_unpacked,their_basis,their_unpacked,index
0,7,000111,14,001110,3
1,11,001011,22,010110,6
2,13,001101,26,011010,8
3,14,001110,28,011100,9
4,19,010011,38,100110,12
5,21,010101,42,101010,14
6,22,010110,44,101100,15
7,25,011001,50,110010,17
8,26,011010,52,110100,18
9,28,011100,56,111000,19
